# 3.3 Brand classification with transfer learning and angle information. 
This notebooko will train all thirty brands on the data using RESNET50 (transfer learning); based on the crop-conclusion from notebook `3.2`. 

This notebook has a COLAB variant after GPU decided to commit harakiri. - Training for phase 3.3 has moved to there; to fit in the restricted space available on google, trainingdata was pre-processed (cropped to bounding bouxes, resized to resnet shape) using notebook `'../to_colab.ipynb'`. 

In [1]:
import pandas as pd
import os
import sys
sys.path.append('../../utils')
from  configloader import Configloader
import cnn_helpers
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
import tensorflow as tf


from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score, f1_score, cohen_kappa_score

2025-04-22 20:35:20.418577: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## 3.3.1 Preparing the system
- Setting constants to us in this notebook and GPU override. 
- Reading config file - no database connection needed as data comes from CSVs
- Add the `CROP` constant

In [2]:
SHAPE = 224   #required for resnet
BATCH_SIZE = 32     #how big ar teh batches for the online learning part
MAX_EPOCHS = 100    #upper limit of epochs per learning task - 
                    #    NOTE THAT there is early stopping and lr plateau just as in nobteook 2.
CROP = True         #Based on notebook 3.2
BALANCE_TEST = True
PHASE = 'brand phase'

In [3]:
cnn_helpers.system_override()
device = cnn_helpers.system_pick_device()

System override applied - check if GPU is detected
Using GPU for deep learning.


2025-04-22 22:59:22.614345: I external/local_xla/xla/stream_executor/rocm/rocm_executor.cc:920] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-04-22 22:59:25.257440: I external/local_xla/xla/stream_executor/rocm/rocm_executor.cc:920] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-04-22 22:59:25.257539: I external/local_xla/xla/stream_executor/rocm/rocm_executor.cc:920] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero


In [4]:
config = Configloader()

basedir = config.get('settings', 'image_directory')
augmdir = config.get('dir_augmentations', 'subfolder')
augmcsv = config.get('dir_augmentations', 'csv_dir')



augment_base = os.path.join(basedir, augmdir, PHASE)
augment_csv_dump = os.path.join(basedir, augmcsv, PHASE)

logfile = os.path.join(config.get('directories', 'root_dir'), 'logging', 'angle_brand_trainingslog.txt')


## 3.3.2 Data loading
Load the CSV's generated by step 3.1 into memory. 

If KERAS would've been able of using integers; then the 300.000 images per angle would've fit in memory: you'd need about 45GB; with the floats however you need about 4 times more! So here too you'll be required to use batch learning. 

In [5]:
traindata = pd.read_csv(os.path.join(augment_csv_dump, 'traindata_brandphase.csv'))
testdata = pd.read_csv(os.path.join(augment_csv_dump, 'testdata_brandphase.csv'))

In [6]:
angles = traindata['model_label'].unique()

## 3.3.3 Label encoding
Now we need to use full label encoding for all thirty brands.

In [7]:
#make an encoder: 
#   LabelEncoder sorts alphabetically and since all classes are always in the traindata, these numbers
#   can be considered stable.
brands = traindata.brand.unique()
label_encoder = LabelEncoder()
label_encoder.fit(brands)

LabelEncoder()

In [8]:
#apply label encoding on train and test dataframes. 
traindata['y_encoded'] = label_encoder.transform(traindata['brand'])
testdata['y_encoded'] = label_encoder.transform(testdata['brand'])

## 3.3.4 Shuffle data: 
Shuffle train and test data; technically not really needed as the generators yield random indexes, but whatever. 

In [9]:
#shuffle
testdata = cnn_helpers.shuffle_df(testdata)
traindata = cnn_helpers.shuffle_df(traindata)

In [10]:
print(len(testdata))
print(len(traindata))

798122
2400000


## 3.3.5 Starting a training loop: 
Everything is ready to go for training, GPU is set up, data is split, shuffled and encoded, all utilities are made in phase `3.2`. We just need to pay special attention to the loop for training, the aim is to train one angle, save the model and then the next. If the training loop crashes while making the n-th angle, it should not retrain the already finished angles. This is easily implemented using a simple `listdir()` statement. 


In [11]:
angles = traindata.model_label.unique()
angles = ['front']  #Only interested in front for now.
model_dest_dir = os.path.join(os.getcwd(), '..', '..', 'models', 'ALL_brand_models_angled')  #TODO read path from config.
os.makedirs(model_dest_dir, exist_ok=True)


for angle in angles: 
    #check if angle is trained already: 
    name = f'FULL_brand_model-Resnet50_cropped={CROP}_angle={angle}_balancedtest={BALANCE_TEST}.keras'
    if name in os.listdir(model_dest_dir):
        print(f"Skipping {name} -- model already trained")
        continue
    cnn_helpers.write_msg_to_log('Started training model: '+name, logfile)
    view_by_angle_test = testdata.query('model_label==@angle').reset_index()
    #Angle is not trained yet: query it from train and test:
    view_by_angle_train = traindata.query('model_label==@angle').reset_index()
    #reduce validation data; is a big bottleneck in performance, still well over 10% of the trainingset
    #reduction_key = 'brand'
    #factor = round((len(view_by_angle_train)*0.12)/view_by_angle_train[reduction_key].nunique())
    #view_by_angle_test_reduced = cnn_helpers.reducer(view_by_angle_test, factor, reduction_key)
    #Get X,y for train and test of the angle view: 
    X_train, y_train = cnn_helpers.get_X_y(view_by_angle_train, 'y_encoded', ['brand'])
    X_test, y_test = cnn_helpers.get_X_y(view_by_angle_test, 'y_encoded', ['brand'])
    model = cnn_helpers.resnet_learner(X_train, X_test, y_train, y_test, SHAPE, CROP, model_dest_dir, BATCH_SIZE, MAX_EPOCHS, resample_test_to_mean=BALANCE_TEST)
    model.save(os.path.join(model_dest_dir, name))
    cnn_helpers.write_msg_to_log('Finished raining model: '+name, logfile)




Resampling test set to mean of the classes
CHECKPOINT found


2025-04-22 22:59:44.628464: I external/local_xla/xla/stream_executor/rocm/rocm_executor.cc:920] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-04-22 22:59:44.628614: I external/local_xla/xla/stream_executor/rocm/rocm_executor.cc:920] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-04-22 22:59:44.628686: I external/local_xla/xla/stream_executor/rocm/rocm_executor.cc:920] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-04-22 22:59:44.628927: I external/local_xla/xla/stream_executor/rocm/rocm_executor.cc:920] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-04-22 22:59:44.628985: I external/local_xla/xla/stream_executor/rocm/rocm_executor.

Configuration or lr scheduler:
{
    "name": "adamw",
    "learning_rate": 0.0002500000118743628,
    "weight_decay": 0.004,
    "clipnorm": null,
    "global_clipnorm": null,
    "clipvalue": null,
    "use_ema": false,
    "ema_momentum": 0.99,
    "ema_overwrite_frequency": null,
    "loss_scale_factor": null,
    "gradient_accumulation_steps": null,
    "beta_1": 0.9,
    "beta_2": 0.999,
    "epsilon": 1e-07,
    "amsgrad": false
}
Base model layers are frozen. Training the new layers first.
Epoch 1/100


I0000 00:00:1745355589.874239   17798 service.cc:146] XLA service 0x77f408003540 initialized for platform ROCM (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1745355589.874279   17798 service.cc:154]   StreamExecutor device (0): AMD Radeon RX 6700 XT, AMDGPU ISA version: gfx1030
2025-04-22 22:59:49.965638: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1745355593.750106   17798 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


9375/9375 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step - accuracy: 0.5228 - loss: 1.7701
Epoch 1: val_accuracy improved from -inf to 0.50270, saving model to /home/frederic/Documents/automotive_project/notebooks/machine learning notebooks/../../models/ALL_brand_models_angled/disposable_dump_of_brand_model_epoch_01__val_loss_1.8670.keras
Saving optimizer state for phase 0 - epcoh 0.
received epcoh checkpoint_epoch_0 and state 0
9375/9375 ━━━━━━━━━━━━━━━━━━━━ 1301s 138ms/step - accuracy: 0.5228 - loss: 1.7701 - val_accuracy: 0.5027 - val_loss: 1.8670 - learning_rate: 2.5000e-04
Epoch 2/100
9375/9375 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step - accuracy: 0.5340 - loss: 1.7324
Epoch 2: val_accuracy did not improve from 0.50270
Saving optimizer state for phase 0 - epcoh 1.
received epcoh checkpoint_epoch_1 and state 0
9375/9375 ━━━━━━━━━━━━━━━━━━━━ 1279s 136ms/step - accuracy: 0.5340 - loss: 1.7324 - val_accuracy: 0.5020 - val_loss: 1.8703 - learning_rate: 2.5000e-04
Epoch 3/100
9375/9375 ━━━━━━━━━━━━━━━━━━━

## 3.3.6 Notebook completed
